# MeasureX Camera-Only Vision POC — Kaggle GPU test

Runs the **real** M0–M5 camera-only pipeline (detect → segment → metric depth → support plane → L/W/H, plus video stabilization and the benchmark) on an actual box photo/video. No mocked inference.

**Requirements**
- A **GPU runtime**: right panel **Settings → Accelerator → GPU** (T4 x2 or P100) **and Settings → Internet → On** (needed to clone the repo and download weights). The notebook stops if CUDA is missing.
- Models: **SAM 2** (Apache-2.0, commercial OK) for segmentation; **Depth Pro** (Apple-ASCL, **research / benchmark only — not for production**) for metric depth.
- The production-safe depth slot (**Depth Anything V2**) is **not implemented yet** (`get_depth_estimator('depth_anything_v2')` raises `NotImplementedError`), so this test uses Depth Pro.
- No marker-mat / ArUco / reference card / LiDAR / ARCore / ARKit. Does **not** touch the Flutter app.

## 0. What to capture

**Photo** (JPG/PNG): one carton, the **whole** box visible with some **floor/ground around its base**, box filling ~1/3–2/3 of the frame, from a **slight angle** so you can see the top and two sides. Sharp, well lit.

**Video** (MP4, 5–10 s): slowly orbit the **same** box ~30–45°, keeping the whole box + floor in frame, steady.

**Measure the real box** with a tape in **millimetres**: Length = longest horizontal, Width = shorter horizontal, Height = vertical. You'll enter these in the benchmark step.

Why the floor: the pipeline finds the **support plane** for the up-axis and base; a box with no ground around it measures poorly or fails.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Clone the repo (branch `main`)

In [ ]:
# Kaggle working dir is /kaggle/working (needs Internet: Settings → Internet → On)
%cd /kaggle/working
!rm -rf MeasureX
!git clone https://github.com/Shaikhabdulaleem/MeasureX.git
%cd MeasureX/services/vision-poc
!git log --oneline -1

## 3. Install the vision POC + model stack

`depth` extra = torch + Depth Pro (Apple git); `seg` extra = torch + SAM 2 (Meta git). Kaggle already ships a CUDA torch, so this mainly pulls the two model packages. A few minutes.

In [ ]:
!pip install -q -e ".[depth,seg]"

## 4. Verify CUDA (stops here if no GPU)

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU — enable a GPU runtime (see top) and rerun.'
print('CUDA OK:', torch.cuda.get_device_name(0))

## 5. Download model weights (real inference)

Depth Pro → `weights/depth_pro.pt` (research-only license). SAM 2.1 large → `weights/sam2.1_hiera_large.pt` (Apache-2.0). Weights are never committed. The cell **fails clearly** if a download didn't land.

In [ ]:
import os
os.makedirs('weights', exist_ok=True)

# Depth Pro (Apple-ASCL, research/benchmark only) -> weights/depth_pro.pt
!python scripts/download_models.py

# SAM 2.1 large (Apache-2.0)
!wget -q -O weights/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

root = '/kaggle/working/MeasureX/services/vision-poc'
os.environ['MEASUREX_DEPTH_CHECKPOINT'] = f'{root}/weights/depth_pro.pt'
os.environ['MEASUREX_SAM2_CHECKPOINT']  = f'{root}/weights/sam2.1_hiera_large.pt'
os.environ['MEASUREX_SAM2_CONFIG']      = 'configs/sam2.1/sam2.1_hiera_l.yaml'
os.environ['MEASUREX_VISION_OUTPUT_DIR'] = f'{root}/outputs'

for p in [os.environ['MEASUREX_DEPTH_CHECKPOINT'], os.environ['MEASUREX_SAM2_CHECKPOINT']]:
    assert os.path.exists(p) and os.path.getsize(p) > 1_000_000, f'weight missing/too small: {p}'
print('weights OK')

## 6. Measure a box from one photo

Result prints as JSON (L/W/H in mm, confidence, diagnostics) or a typed failure, and debug artifacts are saved under `outputs/`.

If SAM 2's config name errors (version drift), list the installed configs with `!python -c "import sam2, os; print(os.listdir(os.path.join(os.path.dirname(sam2.__file__),'configs','sam2.1')))"` and set `MEASUREX_SAM2_CONFIG` to the matching `configs/sam2.1/<file>.yaml`.

**Kaggle upload:** add your files with **+ Add Data → Upload** (creates a dataset under `/kaggle/input/...`). The next cell finds the first image there.

In [ ]:
import glob
imgs = (glob.glob('/kaggle/input/**/*.jpg', recursive=True)
        + glob.glob('/kaggle/input/**/*.jpeg', recursive=True)
        + glob.glob('/kaggle/input/**/*.png', recursive=True))
assert imgs, 'Add your box photo via + Add Data; it appears under /kaggle/input/'
img = imgs[0]; print('using', img)
!python -m measurex_vision.cli image "$img" --provider depth_pro --segmenter sam2 --debug

In [ ]:
# Show the debug artifacts
from IPython.display import Image as IPyImage, display
import os
for f in ['mask_overlay.png', 'depth_color.png']:
    p = f'outputs/{f}'
    print(p); display(IPyImage(p)) if os.path.exists(p) else print('  (missing)')
print(open('outputs/measurement.json').read()) if os.path.exists('outputs/measurement.json') else None

## 7. Measure from a short video (temporal stabilization)

The pipeline samples frames, measures each, rejects outliers, and reports a median L/W/H with per-dimension spread.

In [ ]:
import glob
vids = glob.glob('/kaggle/input/**/*.mp4', recursive=True) + glob.glob('/kaggle/input/**/*.mov', recursive=True)
assert vids, 'Add your box video via + Add Data; it appears under /kaggle/input/'
vid = vids[0]; print('using', vid)
!python -m measurex_vision.cli video "$vid" --provider depth_pro --segmenter sam2 --debug

## 8. Benchmark against your real measured L/W/H

**Edit `L, W, H`** to your tape-measured millimetres (Length ≥ Width). This builds a manifest from the photo + video and runs the M5 benchmark (per-dimension + CBM error, success / explicit-failure / **false-success** rate, single- vs multi-frame, runtime).

In [ ]:
import csv, os, shutil

L, W, H = 600, 400, 350   # <-- EDIT: your real measured millimetres (length >= width)

base = 'data/benchmark/objects/BOX_001/phoneA'
os.makedirs(base, exist_ok=True)
rows = []
if 'img' in globals() and os.path.exists(img):
    shutil.copy(img, f'{base}/image_001.jpg')
    rows.append(('BOX_001_IMG_001', 'image', 'objects/BOX_001/phoneA/image_001.jpg'))
if 'vid' in globals() and os.path.exists(vid):
    shutil.copy(vid, f'{base}/video_001.mp4')
    rows.append(('BOX_001_VID_001', 'video', 'objects/BOX_001/phoneA/video_001.mp4'))
assert rows, 'Run step 6 and/or 7 first.'

with open('data/benchmark/manifest.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['sample_id','object_id','category','phone_model','capture_type','file_path',
                'gt_length_mm','gt_width_mm','gt_height_mm','lighting','distance_band','notes'])
    for sid, ct, fp in rows:
        w.writerow([sid,'BOX_001','medium','phoneA',ct,fp,L,W,H,'warehouse','near','real box'])

!python scripts/benchmark.py data/benchmark/manifest.csv --false-success-tolerance-pct 10
print(open('outputs/benchmark/summary.json').read())

## 9. Where the results are

Under `services/vision-poc/outputs/`:
- **Per run:** `measurement.json`, `mask_overlay.png`, `depth_color.png`, `depth_raw.npy`, `points.npy`, `cloud_diagnostics.json`.
- **Benchmark:** `outputs/benchmark/results.csv`, `summary.json`, and the SVG charts.

**Reading the result:** `success: true` with `dimensions` = predicted L/W/H in mm; `confidence_level` is capped below `high` (uncalibrated until a real benchmark). `success: false` with a `failure_code` (e.g. `geometry_degenerate`, `object_not_fully_visible`) means the scene wasn't measurable — recapture per step 0.